# Scenario summary

Numbers only. Figures live in `../plots/`.

All five runs are on MPP's full 132-pair switch table. Sections 23 and 24 of
`MODEL_REFERENCE.md` record why.

In [1]:
import sys
import importlib
sys.path.append("../..")

import common
importlib.reload(common)   # pick up edits to common.py without restarting the kernel

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common import (SCENARIOS, LABELS, COLOURS, PLANTS, ALUMINA_PER_ALUMINIUM,
                    style, emissions, production, production_by_technology, budget,
                    intensity_split, process_emission_factors, anode, power_source,
                    overlaid, panels, year_axis, save)

style()
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Cumulative emissions against the budget

In [2]:
rows = []
budget_total = (budget("smelter").sum() + budget("refinery").sum()) / 1000
for s in SCENARIOS:
    sm = emissions(s, "smelter").sum() / 1000
    rf = emissions(s, "refinery").sum() / 1000
    rows.append({"Scenario": LABELS[s], "Smelting Gt": sm, "Refining Gt": rf,
                 "Combined Gt": sm + rf, "vs budget %": 100 * ((sm + rf) / budget_total - 1)})
print(f"Budget 2020 to 2050: {budget_total:.2f} Gt")
pd.DataFrame(rows).set_index("Scenario").round(2)

Budget 2020 to 2050: 11.99 Gt


,Smelting Gt,Refining Gt,Combined Gt,vs budget %
Scenario,,,,
Business as usual,27.40,3.72,31.12,159.49
"MPP grid, inert anode locked",11.14,2.06,13.20,10.04
"SBTi grid, inert anode locked",10.25,2.04,12.28,2.41
"MPP grid, inert anode unlocked",10.24,2.04,12.29,2.43
"SBTi grid, inert anode unlocked",10.07,2.06,12.13,1.13


## Emissions intensity, process against electricity

The split is explained in `../plots/03_intensity_process_vs_electricity.ipynb`.

In [3]:
split = {s: intensity_split(s) for s in SCENARIOS}
years = [2025, 2030, 2035, 2040, 2045, 2050]
for component in ["Process", "Electricity", "Total"]:
    print(f"{component} intensity, tCO2e per tonne aluminium")
    display(pd.DataFrame({LABELS[s]: split[s][component] for s in SCENARIOS}).loc[years].round(2))

Process intensity, tCO2e per tonne aluminium


,Business as usual,"MPP grid, inert anode locked","SBTi grid, inert anode locked","MPP grid, inert anode unlocked","SBTi grid, inert anode unlocked"
year,,,,,
2025,3.74,3.74,3.75,3.75,3.73
2030,3.71,3.45,3.44,3.28,3.35
2035,3.38,1.59,1.91,1.71,2.06
2040,3.24,0.73,0.92,0.73,1.50
2045,3.14,0.53,0.48,0.50,0.96
2050,3.01,0.50,0.43,0.50,0.70


Electricity intensity, tCO2e per tonne aluminium


,Business as usual,"MPP grid, inert anode locked","SBTi grid, inert anode locked","MPP grid, inert anode unlocked","SBTi grid, inert anode unlocked"
year,,,,,
2025,9.71,8.41,8.26,8.36,8.09
2030,9.68,7.08,6.96,6.94,7.02
2035,9.68,2.46,1.21,1.96,1.08
2040,9.52,2.01,0.89,1.07,0.28
2045,10.25,1.53,0.72,0.51,0.07
2050,10.66,1.08,0.73,0.05,0.06


Total intensity, tCO2e per tonne aluminium


,Business as usual,"MPP grid, inert anode locked","SBTi grid, inert anode locked","MPP grid, inert anode unlocked","SBTi grid, inert anode unlocked"
year,,,,,
2025,13.45,12.14,12.01,12.12,11.82
2030,13.39,10.53,10.40,10.22,10.37
2035,13.06,4.04,3.12,3.68,3.13
2040,12.76,2.74,1.80,1.80,1.79
2045,13.39,2.06,1.20,1.00,1.03
2050,13.67,1.57,1.15,0.55,0.76


## 2050 power source mix, against MPP's published 1.5DS

In [4]:
CATEGORY = {"Coal": "Fossil fuel", "Natural Gas": "Fossil fuel",
            "Coal+CCS": "Fossil fuel with capture", "Natural Gas+CCS": "Fossil fuel with capture",
            "Grid": "Grid", "PPA+Grid": "Power purchase agreement",
            "Hydro": "Hydro", "Small Modular Reactor": "Nuclear"}
ORDER = ["Fossil fuel", "Fossil fuel with capture", "Grid",
         "Power purchase agreement", "Hydro", "Nuclear"]

def mix_share_2050(scenario):
    p = production_by_technology(scenario, "smelter")
    p.columns = [CATEGORY[power_source(c)] for c in p.columns]
    m = p.T.groupby(level=0).sum().T
    for c in ORDER:
        if c not in m.columns:
            m[c] = 0.0
    row = m[ORDER].loc[2050]
    return 100 * row / row.sum()

published = pd.read_excel("../../../mpp_aluminium_net_zero_outputs.xlsx",
                          sheet_name="Annual_production_volume_Mt_df")
pub = published[(published.scenario == "1.5DS") & (published.plant_type == "Smelter")
                & (published.year == 2050)].copy()
pub["cat"] = [CATEGORY[power_source(t)] for t in pub.technology]
pub_share = 100 * pub.groupby("cat").value.sum().reindex(ORDER).fillna(0)
pub_share = pub_share / pub_share.sum() * 100

table = pd.DataFrame({"MPP published 1.5DS": pub_share,
                      **{LABELS[s]: mix_share_2050(s) for s in SCENARIOS}}).T.round(1)
table["deviation from published"] = (table - table.loc["MPP published 1.5DS"]).abs().sum(axis=1)
table

,Fossil fuel,Fossil fuel with capture,Grid,Power purchase agreement,Hydro,Nuclear,deviation from published
MPP published 1.5DS,0.00,48.30,9.20,9.40,10.00,23.20,0.00
Business as usual,80.90,0.90,9.40,0.00,8.70,0.00,162.40
"MPP grid, inert anode locked",8.40,5.40,34.90,25.20,10.90,15.10,101.80
"SBTi grid, inert anode locked",8.40,6.10,51.90,12.70,9.70,11.20,108.90
"MPP grid, inert anode unlocked",0.00,5.90,38.70,25.50,9.70,20.10,91.40
"SBTi grid, inert anode unlocked",0.00,4.90,70.50,14.90,9.70,0.00,133.70


## Milestone years

No new investment is the first year after which unabated capacity never rises again. Phase out is the first year at or below 1% of the 2020 level, with 2050 as a backstop. Both follow the definitions in choices 25 to 27.

In [5]:
GROUPS = {"digester": (["Coal-Boiler", "Oil-Boiler", "Gas-Boiler"], "refinery", 0),
          "calciner": (["Gas-Calciner", "Oil-Calciner"], "refinery", 1),
          "anode":    (["Carbon Anode"], "smelter", 0)}

def unabated_series(scenario, plant, unabated, part):
    p = production_by_technology(scenario, plant)
    keep = [c for c in p.columns if c.split(" + ")[part] in unabated]
    return p[keep].sum(axis=1)

def milestones(series):
    base = series.loc[2020]
    if base <= 0:
        return None, None
    rises = [y for y in range(2021, 2051) if series[y] > series[y - 1] + 1e-9]
    no_new = (max(rises) + 1) if rises else 2021
    below = [y for y in range(2020, 2051) if series[y] <= 0.01 * base]
    return no_new, (below[0] if below else 2050)

rows = []
for s in SCENARIOS:
    for name, (un, plant, part) in GROUPS.items():
        nn, po = milestones(unabated_series(s, plant, un, part))
        rows.append({"Scenario": LABELS[s], "Asset group": name,
                     "No new investment": nn, "Phase out": po})
pd.DataFrame(rows).pivot(index="Asset group", columns="Scenario",
                         values=["No new investment", "Phase out"])

No new investment                                                                                                                                   Phase out                                                              \
Scenario    Business as usual MPP grid, inert anode locked MPP grid, inert anode unlocked SBTi grid, inert anode locked SBTi grid, inert anode unlocked Business as usual MPP grid, inert anode locked MPP grid, inert anode unlocked   
Asset group                                                                                                                                                                                                                             
anode                    2041                         2031                           2031                          2031                            2031              2050                         2038                           2041   
calciner                 2051                         2051                           2051                          2042                            2050              2050                         2050                           2050   
digester                 2051                         2037                           2032                          2029                            2032              2050                         2050                           2050   

                                                                           
Scenario    SBTi grid, inert anode locked SBTi grid, inert anode unlocked  
Asset group                                                                
anode                                2043                            2050  
calciner                             2050                            2050  
digester                             2050                            2050